# Dataclasses

AI loves dataclasses because they reduce boilerplate. Let's understand this modern Python feature.

## The Problem Dataclasses Solve

Traditional classes have lots of repetitive code:

In [ ]:
# Traditional class - lots of boilerplate!
class PersonOld:
    def __init__(self, name, age, email):
        self.name = name
        self.age = age
        self.email = email
    
    def __repr__(self):
        return f"Person(name={self.name!r}, age={self.age}, email={self.email!r})"
    
    def __eq__(self, other):
        if not isinstance(other, PersonOld):
            return False
        return (self.name == other.name and 
                self.age == other.age and 
                self.email == other.email)

p1 = PersonOld("Alice", 30, "alice@example.com")
print(p1)

In [ ]:
# With dataclass - much cleaner!
from dataclasses import dataclass

@dataclass
class Person:
    name: str
    age: int
    email: str

# __init__, __repr__, __eq__ are auto-generated!
p1 = Person("Alice", 30, "alice@example.com")
p2 = Person("Alice", 30, "alice@example.com")

print(p1)           # Nice repr
print(p1 == p2)     # Equality works

## Default Values

In [ ]:
from dataclasses import dataclass, field
from typing import List

@dataclass
class User:
    name: str
    email: str
    age: int = 0                           # Simple default
    active: bool = True                     # Simple default
    tags: List[str] = field(default_factory=list)  # Mutable default!

user1 = User("Alice", "alice@example.com")
user2 = User("Bob", "bob@example.com", age=25, active=False)

print(user1)
print(user2)

# Tags are separate lists (not shared!)
user1.tags.append("admin")
print(f"User1 tags: {user1.tags}")
print(f"User2 tags: {user2.tags}")  # Empty, not affected

## Important: `field(default_factory=...)` for Mutables

**Never use mutable defaults directly!**

In [ ]:
from dataclasses import dataclass, field

# WRONG - this would share the same list!
# @dataclass
# class BadClass:
#     items: list = []  # ValueError!

# RIGHT - use default_factory
@dataclass
class GoodClass:
    items: list = field(default_factory=list)
    config: dict = field(default_factory=dict)
    
obj = GoodClass()
print(f"Items: {obj.items}, Config: {obj.config}")

## Frozen Dataclasses (Immutable)

In [ ]:
@dataclass(frozen=True)
class Point:
    x: float
    y: float

p = Point(1.0, 2.0)
print(p)

# Can use as dictionary key (hashable)
points = {p: "origin-ish"}
print(f"Dict with Point key: {points}")

# Cannot modify!
try:
    p.x = 5.0
except Exception as e:
    print(f"Error: {type(e).__name__}: {e}")

## Post-Init Processing

In [ ]:
@dataclass
class Rectangle:
    width: float
    height: float
    area: float = field(init=False)  # Not in __init__, computed
    
    def __post_init__(self):
        """Called after __init__ - do validation or compute values."""
        if self.width <= 0 or self.height <= 0:
            raise ValueError("Dimensions must be positive")
        self.area = self.width * self.height

rect = Rectangle(10, 5)
print(f"Rectangle: {rect}")
print(f"Area: {rect.area}")

try:
    bad = Rectangle(-1, 5)
except ValueError as e:
    print(f"Validation error: {e}")

## Dataclass Options

In [ ]:
# All the options
@dataclass(
    init=True,       # Generate __init__ (default True)
    repr=True,       # Generate __repr__ (default True)
    eq=True,         # Generate __eq__ (default True)
    order=False,     # Generate <, <=, >, >= (default False)
    frozen=False,    # Make immutable (default False)
    slots=False,     # Use __slots__ for memory efficiency (3.10+)
)
class Example:
    value: int

# Practical example with ordering
@dataclass(order=True)
class Version:
    major: int
    minor: int
    patch: int

versions = [
    Version(2, 0, 0),
    Version(1, 9, 5),
    Version(1, 10, 0),
]

print("Sorted versions:")
for v in sorted(versions):
    print(f"  {v.major}.{v.minor}.{v.patch}")

## Dataclass vs NamedTuple vs Regular Class

| Feature | `@dataclass` | `NamedTuple` | Regular Class |
|---------|-------------|--------------|---------------|
| Mutable | Yes (default) | No | Yes |
| Auto `__init__` | Yes | Yes | No |
| Auto `__repr__` | Yes | Yes | No |
| Auto `__eq__` | Yes | Yes | No |
| Inheritance | Full | Limited | Full |
| Methods | Yes | Yes | Yes |
| Memory | Normal | Less | Normal |

In [ ]:
# NamedTuple alternative (immutable, tuple-like)
from typing import NamedTuple

class Coordinate(NamedTuple):
    x: float
    y: float
    label: str = "unnamed"

coord = Coordinate(1.0, 2.0, "origin")
print(coord)
print(f"x={coord.x}, y={coord.y}")  # Named access
print(f"As tuple: {tuple(coord)}")   # Tuple access

## AI Code Pattern: Config/Settings Classes

In [ ]:
from dataclasses import dataclass, field
from typing import Optional, List

@dataclass
class AppConfig:
    """Application configuration - common AI-generated pattern."""
    app_name: str
    debug: bool = False
    port: int = 8000
    host: str = "localhost"
    allowed_origins: List[str] = field(default_factory=lambda: ["*"])
    database_url: Optional[str] = None
    
    def is_production(self) -> bool:
        return not self.debug

# Development config
dev_config = AppConfig("MyApp", debug=True)
print(f"Dev: {dev_config}")

# Production config
prod_config = AppConfig(
    app_name="MyApp",
    debug=False,
    port=80,
    host="0.0.0.0",
    database_url="postgresql://..."
)
print(f"Prod: {prod_config}")

## Summary

| Feature | Usage |
|---------|-------|
| `@dataclass` | Auto-generate `__init__`, `__repr__`, `__eq__` |
| Default values | `field: type = default` |
| Mutable defaults | `field(default_factory=list)` |
| Computed fields | `field(init=False)` + `__post_init__` |
| Immutable | `@dataclass(frozen=True)` |
| Sortable | `@dataclass(order=True)` |

## Next Up

Common OOP patterns you'll see in AI-generated code.

Continue to: [OOP Patterns](04-oop-patterns.ipynb)